## 1. Extração (Camada Bronze)
Importação da base de litígios de consumo e identificação inicial dos problemas de tipagem e valores nulos.

In [7]:
# Instala as dependências do kernel do notebook se ainda não estiverem disponíveis
!python -m pip install pandas numpy

import pandas as pd
import numpy as np

# Carrega os dados brutos
df_bronze = pd.read_csv('../01_bronze/reclamacoes_raw.csv', encoding='latin1')

# Exibe as informações de tipagem e nulos
display(df_bronze.info())
display(df_bronze.head())
import numpy as np

# Carrega os dados brutos
df_bronze = pd.read_csv('../01_bronze/reclamacoes_raw.csv', encoding='latin1')

# Exibe as informações de tipagem e nulos
display(df_bronze.info())
display(df_bronze.head())

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id_chamado        50 non-null     int64
 1   data_registro     50 non-null     str  
 2   empresa           50 non-null     str  
 3   uf                50 non-null     str  
 4   problema          50 non-null     str  
 5   status_resolucao  46 non-null     str  
 6   nota_consumidor   39 non-null     str  
dtypes: int64(1), str(6)
memory usage: 2.9 KB



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


None

,id_chamado,data_registro,empresa,uf,problema,status_resolucao,nota_consumidor
0,1001,15/01/2026,American Airlines,SP,Atraso de Voo,Resolvido,4
1,1002,2026-01-16,MaisCredit Consultoria,sp,Cobrança Indevida,Não Resolvido,1
2,1003,17/01/2026,Universidade São Judas,SP,Taxa Abusiva,Resolvido,NaN
3,1004,18-01-2026,AMERICAN AIRLINES,RJ,Extravio de Bagagem,NaN,3
4,1005,19/01/2026,SPPREV,SP,Atraso no Pagamento,Em Andamento,NaN


<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id_chamado        50 non-null     int64
 1   data_registro     50 non-null     str  
 2   empresa           50 non-null     str  
 3   uf                50 non-null     str  
 4   problema          50 non-null     str  
 5   status_resolucao  46 non-null     str  
 6   nota_consumidor   39 non-null     str  
dtypes: int64(1), str(6)
memory usage: 2.9 KB


None

,id_chamado,data_registro,empresa,uf,problema,status_resolucao,nota_consumidor
0,1001,15/01/2026,American Airlines,SP,Atraso de Voo,Resolvido,4
1,1002,2026-01-16,MaisCredit Consultoria,sp,Cobrança Indevida,Não Resolvido,1
2,1003,17/01/2026,Universidade São Judas,SP,Taxa Abusiva,Resolvido,NaN
3,1004,18-01-2026,AMERICAN AIRLINES,RJ,Extravio de Bagagem,NaN,3
4,1005,19/01/2026,SPPREV,SP,Atraso no Pagamento,Em Andamento,NaN


## 2. Tratamento de Strings e Entidades (Camada Silver)
Unificação dos diferentes nomes de registro das partes processuais (réus) e correção de espaços em branco.

In [3]:
df_silver = df_bronze.copy()

# Remove espaços vazios e padroniza a UF para maiúsculo
df_silver['uf'] = df_silver['uf'].str.strip().str.upper()

# Padroniza os nomes das empresas para agrupar corretamente as métricas
df_silver['empresa'] = df_silver['empresa'].str.strip().str.upper()

# Dicionário de padronização de partes (réus comuns em varas cíveis/consumidor)
mapeamento_empresas = {
    'AMERICAN AIRLINES LTDA': 'AMERICAN AIRLINES',
    'MAIS CREDIT': 'MAISCREDIT',
    'MAISCREDIT CONSULTORIA': 'MAISCREDIT',
    'SÃO PAULO PREVIDÊNCIA': 'SPPREV',
    'UNIV. SÃO JUDAS': 'UNIVERSIDADE SÃO JUDAS'
}

df_silver['empresa'] = df_silver['empresa'].replace(mapeamento_empresas)

display(df_silver['empresa'].value_counts())

empresa
AMERICAN AIRLINES         15
MAISCREDIT                15
UNIVERSIDADE SÃO JUDAS    10
SPPREV                    10
Name: count, dtype: int64

## 3. Conversão de Datas
A coluna `data_registro` tem múltiplos formatos estruturais (ex: brasileiro com barras, ISO com traços). A conversão para o tipo nativo `datetime` é fundamental para habilitar análises de inteligência de tempo (YTD, MoM) no Power BI.

In [4]:
# O parâmetro dayfirst garante que 01/02/2026 seja lido como 1º de fevereiro, e format='mixed' lida com as variações
df_silver['data_registro'] = pd.to_datetime(df_silver['data_registro'], format='mixed', dayfirst=True)

display(df_silver[['id_chamado', 'data_registro']].head())

,id_chamado,data_registro
0,1001,2026-01-15
1,1002,2026-01-16
2,1003,2026-01-17
3,1004,2026-01-18
4,1005,2026-01-19


## 4. Tratamento numérico e de nulos
Resolução de anomalias na nota do consumidor (strings inseridas por erro humano e valores fora da escala de 1 a 5) e preenchimento de status vazios.

In [5]:
# Preenche os status em branco com um valor rastreável
df_silver['status_resolucao'] = df_silver['status_resolucao'].fillna('Não Informado')

# Substitui o erro de digitação por extenso para número e converte a coluna para float
df_silver['nota_consumidor'] = df_silver['nota_consumidor'].replace('Dez', 10)
df_silver['nota_consumidor'] = pd.to_numeric(df_silver['nota_consumidor'])

# Trata outliers (notas fora do padrão de 1 a 5) substituindo-os por nulo para não distorcer a média
df_silver.loc[(df_silver['nota_consumidor'] < 1) | (df_silver['nota_consumidor'] > 5), 'nota_consumidor'] = np.nan

display(df_silver.describe())

,id_chamado,data_registro,nota_consumidor
count,50.00000,50,37.000000
mean,1025.50000,2026-02-17 20:38:24,3.432432
min,1001.00000,2026-01-15 00:00:00,1.000000
25%,1013.25000,2026-01-27 06:00:00,2.000000
50%,1025.50000,2026-02-09 12:00:00,4.000000
75%,1037.75000,2026-02-22 18:00:00,5.000000
max,1050.00000,2026-12-02 00:00:00,5.000000
std,14.57738,NaN,1.482126


## 5. Exportação (Camada Silver finalizada)
O dataset higienizado é exportado para alimentar o banco de dados relacional, onde será modelado em Fato e Dimensões.

In [6]:
# Salva o arquivo limpo, garantindo que o index não seja exportado como uma coluna extra
caminho_saida = 'reclamacoes_silver.csv'
df_silver.to_csv(caminho_saida, index=False, encoding='utf-8')

print(f"Pipeline Silver concluído. Dados salvos em: {caminho_saida}")

Pipeline Silver concluído. Dados salvos em: reclamacoes_silver.csv
